In [34]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load environment variables
load_dotenv()

# Configure Groq
api_key = os.getenv("GROQ_API_KEY")
client = Groq(api_key=api_key)

# Send a message
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Hello, how are you?"}],
)

print(response.choices[0].message.content)

Hello. I'm functioning properly, thanks for asking.


In [35]:
def llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


In [36]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

I'm happy to help, but I'm a large language model, I don't have information about a specific course. Could you please provide more context or details about the course you're interested in joining? Such as the course name, institution, or any other relevant information? This will help me assist you more effectively.


In [37]:
context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.

edit on GitHub
#Cloud alternatives with GPU
Check the quota and reset cycle carefully. Is the free hours limit per month or per week? Usually, if you change the configuration, the free hours quota might also be adjusted, or it might be billed separately.

Potential options include:

Google Colab
Kaggle
Databricks (possibly)
Consider using GPTs to discover more options. Be aware that some platforms might have restrictions on what you can and cannot install, so ensure to read what is included in the free vs paid tier.
'''

In [38]:
prompt = f'''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
'''

In [39]:
question = 'I just discovered the course. Can I join now?'
answer = llm(prompt)
print(answer)

You just discovered the course and you want to know if you can join now. From the context, the answer is yes, you can join now, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [40]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)


In [41]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [42]:
courses_raw

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255}]

In [43]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

In [44]:
documents[1110]

{'id': 'bd885d3d4d',
 'course': 'mlops-zoomcamp',
 'section': 'Module 3: Orchestration',
 'question': 'How can I integrate my Dockerized ML model into a Mage pipeline?',
 'answer': "The most effective approach is to have your Docker container serve the model via an HTTP API (FastAPI or Flask), then call that API from a custom Python block in Mage.\n\nIn your Docker container:\n\n- Wrap your model's prediction logic in an API. With FastAPI, create a `/predict` endpoint that accepts input data and returns the model's output.\n- Build and run the container. For local development, run both your model's container and the Mage container with `docker-compose` on the same Docker network so they can reach each other by service name.\n\nIn your Mage pipeline:\n\n- Add a custom Python block (a transformer or data loader).\n- Inside the block, use `requests` to send your input data to the model's `/predict` endpoint.\n- Process the returned predictions and pass them downstream — to a database, ano

In [45]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)
index.fit(documents) 

In [46]:
search_results= index.search(question,
                             boost_dict={'question': 2.0, 'section': 0.5},
                              filter_dict={'course': 'llm-zoomcamp'})

In [47]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}
    return index.search(question,
                             boost_dict={'question': 2.0, 'section': 0.5},
                              filter_dict={'course': 'llm-zoomcamp'})

In [48]:
search_results = search(question)

In [49]:
search_results 

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [50]:
INSTRUCTIONS =f'''
Your task is to answer questions from the course participants
based on the provided context.

Provide only the answer without repeating the question.
Use the context to find relevant information and provide a concise, direct answer.
If the answer is not found in the context, say "I don't know". 
Do not make up answers. Keep your response brief.
'''

In [51]:
USER_PROMPT_TEMPLATE ='''
Question: {question}

Context:{context}
'''

In [52]:
def build_context(search_results) :
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: '+ doc['question' ])
        lines.append('A: '+ doc['answer'])
        lines.append('')

    return '\n'.join(lines). strip()

In [53]:
print(build_context(search_results))

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [54]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question=question, context=context)
    return prompt.strip()


In [55]:
prompt = build_prompt(question, search_results)
print(prompt)

Question: I just discovered the course. Can I join now?

Context:General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

In [56]:

response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )


In [57]:
response

ChatCompletion(id='chatcmpl-fb73d722-1517-46ee-929a-ed0e94acc168', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Yes, you can join the course now. However, please note that if you want to receive a certificate, you need to submit your project while we're still accepting submissions.", role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1779127539, model='llama-3.1-8b-instant', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_4387d3edbb', usage=CompletionUsage(completion_tokens=36, prompt_tokens=991, total_tokens=1027, completion_time=0.077205292, completion_tokens_details=None, prompt_time=0.15711946, prompt_tokens_details=None, queue_time=0.112528689, total_time=0.234324752), usage_breakdown=None, x_groq=XGroq(id='req_01kry47n35fww9ddqfdsmvtbpj', debug=None, seed=1575825921, usage=None))

In [58]:
response.choices[0].message.content

"Yes, you can join the course now. However, please note that if you want to receive a certificate, you need to submit your project while we're still accepting submissions."

In [59]:
print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-fb73d722-1517-46ee-929a-ed0e94acc168",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Yes, you can join the course now. However, please note that if you want to receive a certificate, you need to submit your project while we're still accepting submissions.",
        "role": "assistant",
        "annotations": null,
        "executed_tools": null,
        "function_call": null,
        "reasoning": null,
        "tool_calls": null
      }
    }
  ],
  "created": 1779127539,
  "model": "llama-3.1-8b-instant",
  "object": "chat.completion",
  "mcp_list_tools": null,
  "service_tier": "on_demand",
  "system_fingerprint": "fp_4387d3edbb",
  "usage": {
    "completion_tokens": 36,
    "prompt_tokens": 991,
    "total_tokens": 1027,
    "completion_time": 0.077205292,
    "completion_tokens_details": null,
    "prompt_time": 0.15711946,
    "prompt_tokens_details": null,
    "queue_time":

In [60]:
response.usage

CompletionUsage(completion_tokens=36, prompt_tokens=991, total_tokens=1027, completion_time=0.077205292, completion_tokens_details=None, prompt_time=0.15711946, prompt_tokens_details=None, queue_time=0.112528689, total_time=0.234324752)

In [61]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.prompt_tokens * input_price +
    response.usage.completion_tokens * output_price
)

print(cost)

0.00090525


In [62]:
message_history = [
    {'role': 'system', 'content': INSTRUCTIONS},
    {'role': 'user', 'content': prompt}
]

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=message_history
)

In [63]:
response.choices[0].message.content

'Yes, you can join now.'

In [64]:
def llm(instructions, user_prompt, model="llama-3.1-8b-instant"):
    message_history = [
        {'role': 'system', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=message_history
    )
    return response.choices[0].message.content



In [65]:
def rag(query, model='llama-3.1-8b-instant'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [66]:
answer = rag("How do I get a certificate?")
print(answer)

To get a certificate. You need to finish the course with a "live" cohort, pass the Capstone project, and peer-review three projects.
